In [136]:
import pandas as pd

In [137]:
no_rag_no_auto_df = pd.read_csv("./no_rag_no_auto_new.csv")
rag_no_auto_df = pd.read_csv("./rag_no_auto.csv")
rag_auto_df = pd.read_csv("./rag_auto.csv")


print(no_rag_no_auto_df.shape)
print(rag_no_auto_df.shape)
print(rag_auto_df.shape)

(257, 13)
(257, 13)
(257, 13)


In [138]:
# Filter out rows where 'error' is in 'LLM Generated SQL Query' lowered
no_rag_no_auto_df = no_rag_no_auto_df[~no_rag_no_auto_df['LLM Generated SQL Query'].str.lower().str.contains('error')]
rag_no_auto_df = rag_no_auto_df[~rag_no_auto_df['LLM Generated SQL Query'].str.lower().str.contains('error')]
rag_auto_df = rag_auto_df[~rag_auto_df['LLM Generated SQL Query'].str.lower().str.contains('error')]


In [139]:
# add a new column 'id' which is append of 'Natural Language Query' and 'SQL Query'
no_rag_no_auto_df['id'] = no_rag_no_auto_df['Natural Language Query'] + ' ' + no_rag_no_auto_df['SQL Query']
rag_no_auto_df['id'] = rag_no_auto_df['Natural Language Query'] + ' ' + rag_no_auto_df['SQL Query']
rag_auto_df['id'] = rag_auto_df['Natural Language Query'] + ' ' + rag_auto_df['SQL Query']




In [140]:
# drop duplicates based on 'Natural Language Query' and 'SQL Query'
no_rag_no_auto_df = no_rag_no_auto_df.drop_duplicates(subset=['id'])
rag_no_auto_df = rag_no_auto_df.drop_duplicates(subset=['id'])
rag_auto_df = rag_auto_df.drop_duplicates(subset=['id'])



In [141]:
print(no_rag_no_auto_df.shape)
print(rag_no_auto_df.shape)
print(rag_auto_df.shape)

(218, 14)
(220, 14)
(211, 14)


In [142]:
rag_no_auto_df.columns


Index(['Natural Language Query', 'SQL Query', 'Schema',
       'Top 5 Entries of Table', 'Source_Sheet', 'Table_Names',
       'Syntactically_Correct', 'Logically_Correct', 'LLM Generated SQL Query',
       'LLM_Query_Execution_Results', 'Correct_Query_Execution_Results',
       'Correct_Query_Syntactical_Correctness', 'LLM_Generated_Summary', 'id'],
      dtype='object')

In [143]:
# Find intersection of 'id', 'Natural Language Query', 'SQL Query' column among all dataframes
common_queries = set(no_rag_no_auto_df['id'])
for df in [rag_no_auto_df, rag_auto_df]:
    common_queries.intersection_update(df['id'])

print(len(common_queries))


202


In [144]:
#  filter dataframes so that they all contain the intersection of (Natural Language Query, SQL Query) pairs







In [145]:
# filter the dataframes to only include the common queries
no_rag_no_auto_df = no_rag_no_auto_df[no_rag_no_auto_df['id'].isin(common_queries)]
rag_no_auto_df = rag_no_auto_df[rag_no_auto_df['id'].isin(common_queries)]
rag_auto_df = rag_auto_df[rag_auto_df['id'].isin(common_queries)]




In [146]:
# sort the dataframes by 'Natural Language Query' column
no_rag_no_auto_df = no_rag_no_auto_df.sort_values(by='id')
rag_no_auto_df = rag_no_auto_df.sort_values(by='id')
rag_auto_df = rag_auto_df.sort_values(by='id')




In [147]:
print(no_rag_no_auto_df.shape)
print(rag_no_auto_df.shape)
print(rag_auto_df.shape)





(202, 14)
(202, 14)
(202, 14)


In [148]:
#  compare syntacticall and logical correctness numbers in a table, using 'Syntactically_Correct', 'Logically_Correct'

# Calculate correctness counts for each configuration
metrics = {
    'No RAG, No Auto': {
        'Syntactically Correct': no_rag_no_auto_df['Syntactically_Correct'].sum(),
        'Logically Correct': no_rag_no_auto_df['Logically_Correct'].sum(),
        'Syntactical accuracy': 100*no_rag_no_auto_df['Syntactically_Correct'].sum()/len(no_rag_no_auto_df),
        'Logical accuracy': 100*no_rag_no_auto_df['Logically_Correct'].sum()/len(no_rag_no_auto_df) 
    },
    'RAG, No Auto': {
        'Syntactically Correct': rag_no_auto_df['Syntactically_Correct'].sum(),
        'Logically Correct': rag_no_auto_df['Logically_Correct'].sum(),
        'Syntactical accuracy': 100*rag_no_auto_df['Syntactically_Correct'].sum()/len(rag_no_auto_df),
        'Logical accuracy': 100*rag_no_auto_df['Logically_Correct'].sum()/len(rag_no_auto_df)
    },
    'RAG, Auto': {
        'Syntactically Correct': rag_auto_df['Syntactically_Correct'].sum(),
        'Logically Correct': rag_auto_df['Logically_Correct'].sum(),
        'Syntactical accuracy': 100*rag_auto_df['Syntactically_Correct'].sum()/len(rag_auto_df),
        'Logical accuracy': 100*rag_auto_df['Logically_Correct'].sum()/len(rag_auto_df)
    }
}

# Create a summary DataFrame from the metrics dictionary
comparison_table = pd.DataFrame(metrics).T # Transpose to get configurations as rows

# Display the table
print(comparison_table)

# Optional: Add a total row
comparison_table.loc['Total Queries'] = len(common_queries) # Should be 128 if filtering worked as expected
print("\\nComparison Table with Total:")
print(comparison_table)



                 Syntactically Correct  Logically Correct  \
No RAG, No Auto                  159.0               11.0   
RAG, No Auto                     171.0                2.0   
RAG, Auto                        176.0                5.0   

                 Syntactical accuracy  Logical accuracy  
No RAG, No Auto             78.712871          5.445545  
RAG, No Auto                84.653465          0.990099  
RAG, Auto                   87.128713          2.475248  
\nComparison Table with Total:
                 Syntactically Correct  Logically Correct  \
No RAG, No Auto                  159.0               11.0   
RAG, No Auto                     171.0                2.0   
RAG, Auto                        176.0                5.0   
Total Queries                    202.0              202.0   

                 Syntactical accuracy  Logical accuracy  
No RAG, No Auto             78.712871          5.445545  
RAG, No Auto                84.653465          0.990099  
RAG, Auto   